# 02 — HMM Decoding and State-Specific FC

Decode HMM states, split time series by subject, and compute per-subject/state functional connectivity matrices (soft gamma weighting by default).


In [1]:
import os
from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

    DATA_MODE = CONFIG.get("data_mode", "example")
    N_STATES = CONFIG["n_states"]
    N_TIMEPOINTS = CONFIG["n_timepoints"]
    N_SUBJECTS = CONFIG["n_subjects"]
    METHOD = CONFIG["method"]
    USE_GAMMA = CONFIG["use_gamma"]
    MIN_EFF_N = CONFIG["min_eff_n"]
    MIN_HARD_N = CONFIG["min_hard_n"]
    GRAD = CONFIG["gradient"]

    DATA_ROOT = PROJECT_ROOT / (CONFIG["paths"]["example_root"] if DATA_MODE == "example" else CONFIG["paths"]["study_data_root"])
    OUTPUT_ROOT = PROJECT_ROOT / CONFIG["paths"]["output_root"] / DATA_MODE
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

    print(f"project={PROJECT_ROOT} mode={DATA_MODE} out={OUTPUT_ROOT}")

    import json
    import pickle
    from typing import Dict

    import numpy as np
    import pandas as pd

    with open(OUTPUT_ROOT / "analysis_metadata.json", encoding="utf-8") as f:
        meta = json.load(f)

        K = int(meta["K"])
        method = meta["method"]
        n_timepoints = int(meta["n_timepoints"])
        use_gamma = bool(meta["use_gamma"])
        subjects = meta["subjects"]
        select_networks = np.array(meta["select_networks"])

        roi_dir = OUTPUT_ROOT / "01_roi"
        sort_idx = np.load(roi_dir / "sort_idx.npy")

        if DATA_MODE == "example":
            Y = pd.read_csv(DATA_ROOT / "parcel_timeseries.csv").values[:, sort_idx]
            idx_df = pd.read_csv(DATA_ROOT / "subject_indices.csv")
            decode_npz = np.load(DATA_ROOT / "hmm_decode.npz")
            Gamma = decode_npz["Gamma"]
            vpath = decode_npz["vpath"]
            hmm = None
            print("Example mode: loaded precomputed HMM decode arrays.")
        else:
            from glhmm import statistics

            Y = pd.read_csv(DATA_ROOT / f"all_schaefer_parcel1000_{method}.csv").values[:, sort_idx]
        with open(
            PROJECT_ROOT / CONFIG["paths"]["hmm_model_template"].format(k=K, method=method),
            "rb",
            ) as f:
                hmm = pickle.load(f)
                if not subjects:
                    from bids import BIDSLayout
                    subjects = BIDSLayout(PROJECT_ROOT.parent, validate=False, derivatives=True).get_subjects()
                    indices = statistics.get_indices_timestamp(n_timepoints, len(subjects))

project=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis mode=example out=d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example


Example mode: loaded precomputed HMM decode arrays.


In [2]:
if DATA_MODE != "example":
    Gamma, Xi, _ = hmm.decode(X=None, Y=Y, indices=indices, viterbi=False)
    vpath = hmm.decode(X=None, Y=Y, indices=indices, viterbi=True)

    Y_by_subj = {}
    gamma_by_subj = {}
    states_by_subj = {}
    start = 0
    for subj in subjects:
        n_tp = (
        int(idx_df.loc[idx_df.subject == subj, "n_timepoints"].iloc[0])
        if DATA_MODE == "example"
        else n_timepoints
        )
        Y_by_subj[subj] = Y[start : start + n_tp, :]
        gamma_by_subj[subj] = Gamma[start : start + n_tp, :]
        states_by_subj[subj] = np.argmax(vpath[start : start + n_tp, :], axis=1)
        start += n_tp

        print(f"Prepared {len(subjects)} subjects.")

Prepared 5 subjects.


In [3]:
def compute_weighted_correlation(timeseries: np.ndarray, weights: np.ndarray) -> np.ndarray:
    w = np.clip(weights.astype(float), 0.0, np.inf)
    if np.all(w == 0):
        return np.full((timeseries.shape[1], timeseries.shape[1]), np.nan)
        w = w / (w.sum() + 1e-12)
        mean_w = (timeseries * w[:, None]).sum(axis=0)
        Xc = timeseries - mean_w
        cov_w = (Xc * w[:, None]).T @ Xc
        var = np.diag(cov_w)
        denom = np.sqrt(np.outer(var, var)) + 1e-12
        return np.clip(cov_w / denom, -1.0, 1.0)


def effective_sample_size(weights: np.ndarray) -> float:
    w = weights.astype(float)
    if np.all(w == 0):
        return 0.0
        return (w.sum() ** 2) / ((w ** 2).sum() + 1e-12)


        per_subject_state_FC: Dict[str, Dict[int, np.ndarray]] = {}
        summary_rows = []

        for subj, Y_subj in Y_by_subj.items():
            T_i, n_regions = Y_subj.shape
            per_subject_state_FC[subj] = {}
            if use_gamma:
                gamma = gamma_by_subj[subj]
                for k in range(K):
                    w = gamma[:, k]
                    neff = effective_sample_size(w)
                    FC = (
                    np.full((n_regions, n_regions), np.nan)
                    if neff < MIN_EFF_N
                    else compute_weighted_correlation(Y_subj, w)
                    )
                    per_subject_state_FC[subj][k] = FC
                    summary_rows.append(
                    {"subj": subj, "state": k + 1, "method": "soft", "eff_n_or_n": float(neff)}
                    )
        else:
            z = states_by_subj[subj]
            for k in range(K):
                idx = z == k
                n_used = int(idx.sum())
                FC = (
                np.full((n_regions, n_regions), np.nan)
                if n_used < MIN_HARD_N
                else np.corrcoef(Y_subj[idx], rowvar=False)
                )
                per_subject_state_FC[subj][k] = FC
                summary_rows.append(
                {"subj": subj, "state": k + 1, "method": "hard", "eff_n_or_n": float(n_used)}
                )

                summary_df = pd.DataFrame(summary_rows)
                print(summary_df.head())

                out_dir = OUTPUT_ROOT / "02_state_fc"
                out_dir.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(
                out_dir / "per_subject_state_FC.npz",
                allow_pickle=True,
                per_subject_state_FC=per_subject_state_FC,
                region_names=select_networks,
                )
                summary_df.to_csv(out_dir / "per_subject_state_FC_summary.csv", index=False)
                np.savez_compressed(out_dir / "hmm_decode.npz", Gamma=Gamma, vpath=vpath, K=K, subjects=np.array(subjects))
                print(f"Saved state FC → {out_dir}")

 subj state method eff_n_or_n
0 sub-01 1 soft 81.349999
1 sub-01 2 soft 76.283567
2 sub-01 3 soft 95.242044
3 sub-02 1 soft 84.244873
4 sub-02 2 soft 77.163899
Saved state FC → d:\文件同步\文件\数据分析结果\data_code_figure\gradient_dispersion_mediate_moderate\gradient_analysis\results\example\02_state_fc
